# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\textsuperscript{2} dataset package using the `mlcroissant` library, referencing all entities via their `@id` fields where required.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not as a dictionary)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

print(f"Authors (ids): {[getattr(author, '@id', str(author)) for author in getattr(metadata, 'author', [])]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate the available RecordSets in the dataset by their `@id` and list out their main fields, also referenced via `@id`.

In [ ]:
# List all record sets in the dataset
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # mlcroissant >=0.8.0 supports .record_sets if .recordSet is empty, fallback for older spec
    record_sets = getattr(metadata, 'record_sets', [])

if not record_sets:
    raise Exception('No record sets found in metadata.')

for idx, rs in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', rs)}")
    # Show fields if available
    fields = getattr(rs, 'field', None)
    if not fields and isinstance(rs, dict):
        fields = rs.get('field', None)
    if fields:
        if isinstance(fields, list):
            print("    Fields (by @id):")
            for f in fields:
                field_id = f['@id'] if isinstance(f, dict) and '@id' in f else getattr(f, '@id', f)
                print(f"      - {field_id}")
        else:
            print(f"    Field: {fields}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll collect the `@id`s, load each record set, and preview the data.

In [ ]:
# Collect all record set @id fields
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', rs)
    record_set_ids.append(rs_id)

if not record_set_ids:
    raise Exception("No record sets found in metadata for extraction.")

dataframes = {}
for rs_id in record_set_ids:
    # Iterating over each record and collecting into a dataframe
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"RecordSet @id: {rs_id} : columns = {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"RecordSet @id: {rs_id} has no records.")

# Choose the first available record set for further analysis
main_record_set_id = record_set_ids[0]
if main_record_set_id not in dataframes:
    raise Exception(f"No records found in main record set {main_record_set_id}.")
df_main = dataframes[main_record_set_id]
print(f"Using record set @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. All columns are referenced by their `@id`.

We'll pick a numeric column for demo (such as protein mass or peptide count, etc), filter, normalize, and group.

_**Note:** Replace `<numeric_field_id>` and `<group_field_id>` if you wish to target a specific field. The demonstration uses an available column id._

In [ ]:
# List all available columns in main DataFrame
print('Data columns:', df_main.columns.tolist())

# Example: Try to guess a numeric field from the columns (e.g., 'cr:peptide_count', 'cr:mw', 'cr:coverage')
possible_numeric_ids = [c for c in df_main.columns if any(x in c.lower() for x in ['count', 'mw', 'weight', 'coverage', 'abundance', 'numeric', 'number', 'cr:peptide_count', 'cr:mw', 'cr:coverage'])]
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]
else:
    numeric_field_id = df_main.columns[0]  # fallback
print(f"Selected numeric field for analysis: {numeric_field_id}")

threshold = df_main[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df_main[numeric_field_id]) else 0

# Filter records where numeric_field > threshold
filtered_df = df_main[df_main[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the selected numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Try to find a groupable (categorical) field
possible_group_ids = [c for c in df_main.columns if any(x in c.lower() for x in ['group', 'category', 'type', 'sample', 'class', 'label'])]
if possible_group_ids:
    group_field_id = possible_group_ids[0]
    print(f"Selected group/categorical field: {group_field_id}")
else:
    group_field_id = None

# Group by the group field if it exists
if group_field_id and group_field_id in df_main.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example of a histogram and a boxplot for the main numeric field, as well as a boxplot by group if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of main numeric field
plt.figure(figsize=(8,4))
sns.histplot(df_main[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot of main numeric field
plt.figure(figsize=(4,5))
sns.boxplot(y=df_main[numeric_field_id])
plt.title(f"Boxplot of {numeric_field_id}")
plt.ylabel(numeric_field_id)
plt.show()

# Boxplot by group if group_field_id exists
if group_field_id and group_field_id in df_main.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrates how to load and explore the [FAIR$^2$ protein dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library, referencing all entities by their `@id`.

- We loaded the dataset metadata and record sets programmatically using `mlcroissant`.
- We conducted data inspection, filtering, normalization, grouping, and basic visualizations using field and record set `@id`s.
- This workflow provides a robust, schema-aware approach for scientific and FAIR machine-learning dataset exploration.

Feel free to extend this notebook for further domain-specific analyses.